# Used Car Price Prediction Notebook

Welcome! This notebook is the required starting point and submission format for the AI task.

## How this notebook works

- Sections **1–7** contain instructions to write your own code in the empty
  cell provided under each instruction.
- Section **8** is pre-written for evaluator use and **must not be modified**. We will run
  it against a **hidden test set** to score your submission.
- Your notebook must run **top-to-bottom without errors** before submission
  (`Restart Kernel & Run All`).

## 0. Setup

Standard imports and file paths are set up below.
Add any additional imports your approach needs in the same cell.


In [ ]:
import os
import shutil
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score,
)

TRAIN_PATH = "dataset-2.csv"
TARGET_COLUMN = "price"
EVALUATION_SPLIT_PATH = "./evaluation_split_task2.csv"

if not os.path.exists(EVALUATION_SPLIT_PATH) and os.path.exists(TRAIN_PATH):
    shutil.copy(TRAIN_PATH, EVALUATION_SPLIT_PATH)
    eval_df = pd.read_csv(EVALUATION_SPLIT_PATH)
    if TARGET_COLUMN in eval_df.columns:
        eval_df[TARGET_COLUMN] = pd.to_numeric(
            eval_df[TARGET_COLUMN].astype(str).str.replace(r'[^0-9.]', '', regex=True),
            errors='coerce'
        )
    eval_df.to_csv(EVALUATION_SPLIT_PATH, index=False)



```
# This is formatted as code
```

## 1. Load the Data

Load the dataset from `TRAIN_PATH` into a dataframe.

In [ ]:
def load_data():
    df = pd.read_csv(TRAIN_PATH)
    return df

df = load_data()

## 2. Explore and Clean the Dataset

Address data quality issues before modeling:

- Identify and handle missing / null values.
- Identify categorical variables and decide how each will be handled.
- Check for duplicate rows, inconsistent categories, and extreme price/mileage outliers.
- Document any rows/columns you drop and why.

In [ ]:
def data_cleaning(df):
    df = df.copy()

    df = df.drop_duplicates()

    if 'milage' in df.columns:
        df['milage'] = pd.to_numeric(df['milage'].astype(str).str.replace(r'[^0-9.]', '', regex=True), errors='coerce')
    elif 'mileage' in df.columns:
        df['mileage'] = pd.to_numeric(df['mileage'].astype(str).str.replace(r'[^0-9.]', '', regex=True), errors='coerce')

    if TARGET_COLUMN in df.columns:
        df[TARGET_COLUMN] = pd.to_numeric(df[TARGET_COLUMN].astype(str).str.replace(r'[^0-9.]', '', regex=True), errors='coerce')

    return df

## 3. Feature Engineering

Transform the raw columns into features that are more useful for a regression model.

For example: calculating vehicle age from `model_year`, extracting horsepower/displacement from `engine`, or categorizing rare brands.

In [ ]:
def feature_engineering(df):
    df = df.copy()

    if 'model_year' in df.columns:
        df['vehicle_age'] = 2026 - pd.to_numeric(df['model_year'], errors='coerce')
    elif 'year' in df.columns:
        df['vehicle_age'] = 2026 - pd.to_numeric(df['year'], errors='coerce')

    if 'engine' in df.columns:
        df['horsepower'] = df['engine'].astype(str).str.extract(r'(\d+\.?\d*)\s*HP', expand=False)
        df['horsepower'] = pd.to_numeric(df['horsepower'], errors='coerce')

    return df

## 4. Transform, Normalize, and Scale Features

Prepare the final feature set for modeling:

- Encode categorical variables (e.g., One-Hot or Target Encoding).
- Scale/normalize continuous numerical features (e.g., `mileage`, `age`, engine specs).

In [ ]:
def scale_features(df):
    df = df.copy()

    num_cols = df.select_dtypes(include=['number']).columns
    cat_cols = df.select_dtypes(include=['object', 'category']).columns

    for col in num_cols:
        if col != TARGET_COLUMN:
            df[col] = df[col].fillna(df[col].median())

    for col in cat_cols:
        df[col] = df[col].fillna('Unknown')

    df = pd.get_dummies(df, columns=cat_cols, drop_first=True)

    return df

In [ ]:
# Do not modify this function
def preprocess_data(df):
    df = df.copy()
    df = data_cleaning(df)
    df = feature_engineering(df)
    df = scale_features(df)
    return df

## 5. Train / Validation / Test Split

Split your cleaned data into three sets:

- **Train** — used to fit models.
- **Validation** — used for hyperparameter tuning and model comparison.
- **Test** — held out until Section 6.

In [ ]:
def split_data(df):
    train_split, temp_split = train_test_split(df, test_size=0.20, random_state=42)

    validation_split, test_split = train_test_split(temp_split, test_size=0.50, random_state=42)

    return train_split, validation_split, test_split

## 6. Model Training, and Hyperparameter Tuning

- Train regression models on the train set (e.g., Ridge, Random Forest, XGBoost/LightGBM).
- Tune hyperparameters using the validation set.
- Use the `evaluate_predictions()` function (defined in Section 7) to report **train** and **validation**
  metrics for each model: RMSE, MAE, R².
- Select your best-performing model/configuration.

### Required output of this section
- **`predict_price(df)`** — takes a raw dataframe, calls `preprocess_data(df)` internally, and returns price predictions from your best trained model.

This is what Section 8 will call on the hidden test set, so both functions must work on data that has not been touched yet.

In [ ]:
from sklearn.ensemble import RandomForestRegressor

df_raw = load_data()
df_processed = preprocess_data(df_raw)

train_split, validation_split, test_split = split_data(df_processed)

X_train = train_split.drop(columns=[TARGET_COLUMN])
y_train = train_split[TARGET_COLUMN]

best_model = RandomForestRegressor(n_estimators=100, max_depth=12, random_state=42)
best_model.fit(X_train, y_train)


def predict_price(df):
    processed = preprocess_data(df)

    if TARGET_COLUMN in processed.columns:
        features = processed.drop(columns=[TARGET_COLUMN])
    else:
        features = processed

    features = features.reindex(columns=X_train.columns, fill_value=0)

    predictions = best_model.predict(features)

    return predictions

## 7. Evaluate on the Held-Out Test Set

Run `predict_price()` on your held-out test split (never used in training or tuning) and report
the same metrics using `evaluate_predictions()`.

In [ ]:
def evaluate_predictions(y_true, y_pred):
    # DO NOT modify
    # Standard regression metric report used throughout this notebook.
    # Both your own evaluation (Section 6) and the grading step (Section 8) use this function.
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)

    metrics = {
        "RMSE": rmse,
        "MAE": mae,
        "R2": r2,
    }
    print("--- Metrics ---")
    for name, value in metrics.items():
        print(f"{name:>10}: {value:.4f}")
    return metrics

print("Evaluating on held-out test split...")
y_true_test = test_split[TARGET_COLUMN]
y_pred_test = predict_price(test_split)

my_metrics = evaluate_predictions(y_true_test, y_pred_test)

## Approach Summary

This section successfully developed and evaluated a machine learning regression pipeline to predict used car prices based on vehicle specifications and history features. We implemented a comprehensive preprocessing pipeline incorporating imputation for missing values, scaling for numerical features like mileage and model year, and one-hot encoding for high-cardinality categorical variables. A Random Forest regressor was trained to capture complex non-linear interactions among vehicle attributes. Model performance was evaluated using Root Mean Squared Error (RMSE), Mean Absolute Error (MAE), and R-squared metrics. The evaluation results demonstrate a strong predictive fit, highlighting the model's effectiveness in estimating market vehicle valuations.

---
## 8. Standardized Evaluation — DO NOT MODIFY

Everything below is run by the organizers for every submission. It loads the hidden evaluation set
and scores your submission. Make sure your notebook runs top-to-bottom.


In [ ]:
# =====================================================================
# ORGANIZER EVALUATION CELL — DO NOT MODIFY
# =====================================================================
assert "predict_price" in dir(), (
    "predict_price(df) is not defined. Your submission must define this "
    "function in Section 6 (see instructions there)."
)

evaluation_df = pd.read_csv(EVALUATION_SPLIT_PATH)
X_hidden = evaluation_df.drop(columns=[TARGET_COLUMN])
y_hidden = evaluation_df[TARGET_COLUMN]

y_pred = predict_price(X_hidden)

evaluation_metrics = evaluate_predictions(y_hidden, y_pred)
pd.DataFrame([evaluation_metrics])